# Демонстрация улучшений пайплайна

Notebook демонстрирует шесть ключевых улучшений:
1. Декластеризация Gardner-Knopoff на синтетических данных (до/после)
2. FDR коррекция: гистограмма raw p-values vs adjusted
3. ETAS каталог: визуализация (scatter времени, магнитуда)
4. Частота ложных серий на 10 ETAS каталогах
5. Калибровка порогов: heatmap F1(N_min, n_regions)
6. Тектоническое расстояние v2 с типами границ

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings
warnings.filterwarnings('ignore')

# Глобальные настройки графиков
plt.rcParams.update({
    'figure.dpi': 100,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})
print('Среда готова.')

## 1. Декластеризация Gardner-Knopoff

Создаём синтетический каталог с явным кластером (1 главный толчок + 8 афтершоков) и применяем алгоритм Gardner-Knopoff (1974).

In [ ]:
from src.analysis.declustering import GardnerKnopoffDeclustering, compare_declustering_methods

rng = np.random.default_rng(42)

# ── Синтетический каталог ──
records = []

# Главный толчок M=7.2 в точке (0°, 90°)
records.append({'time': pd.Timestamp('2000-01-15'), 'lat': 0.0, 'lon': 90.0, 'magnitude': 7.2})

# 8 афтершоков: < 15 дней, < 50 км
for i in range(8):
    dt = rng.uniform(0.5, 14)
    dlat = rng.uniform(-0.3, 0.3)
    dlon = rng.uniform(-0.3, 0.3)
    m = float(rng.uniform(5.5, 6.8))
    records.append({'time': pd.Timestamp('2000-01-15') + pd.Timedelta(days=dt),
                    'lat': dlat, 'lon': 90.0 + dlon, 'magnitude': m})

# 15 фоновых событий (далеко, через несколько месяцев)
for i in range(15):
    records.append({'time': pd.Timestamp('2000-01-15') + pd.Timedelta(days=rng.integers(200, 400)),
                    'lat': float(rng.uniform(-60, 60)),
                    'lon': float(rng.uniform(-180, 180)),
                    'magnitude': float(rng.uniform(6.5, 7.5))})

catalog = pd.DataFrame(records).sort_values('time').reset_index(drop=True)

# ── Gardner-Knopoff ──
gk = GardnerKnopoffDeclustering()
mainshocks, aftershocks = gk.decluster(catalog)

print(f'Исходный каталог:   {len(catalog)} событий')
print(f'Главные толчки:     {len(mainshocks)} событий')
print(f'Зависимые события:  {len(aftershocks)} событий')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# До декластеризации
ax = axes[0]
ax.scatter(catalog['lon'], catalog['lat'], c=catalog['magnitude'],
           cmap='YlOrRd', s=80, edgecolors='k', linewidths=0.5, vmin=5, vmax=8)
ax.set_title('До декластеризации\n(%d событий)' % len(catalog), fontsize=13)
ax.set_xlabel('Долгота')
ax.set_ylabel('Широта')
ax.set_xlim(-185, 185)

# После декластеризации
ax = axes[1]
sc = ax.scatter(mainshocks['lon'], mainshocks['lat'], c=mainshocks['magnitude'],
                cmap='YlOrRd', s=100, edgecolors='k', linewidths=0.5, vmin=5, vmax=8,
                label='Главные толчки', zorder=3)
if len(aftershocks) > 0:
    ax.scatter(aftershocks['lon'], aftershocks['lat'], c='steelblue', alpha=0.4,
               s=40, marker='x', label='Афтершоки (удалены)')
ax.set_title('После Gardner-Knopoff\n(%d главных толчков)' % len(mainshocks), fontsize=13)
ax.set_xlabel('Долгота')
ax.legend()
ax.set_xlim(-185, 185)

plt.colorbar(sc, ax=axes, label='Магнитуда', fraction=0.02, pad=0.04)
plt.suptitle('Декластеризация Gardner-Knopoff (1974)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. FDR коррекция: raw vs adjusted p-values

Симулируем тестирование 50 серий: 5 реально значимых (p < 0.001) и 45 случайных.

In [ ]:
from src.analysis.multiple_testing import apply_fdr_bh, apply_bonferroni, apply_bh_to_series

rng2 = np.random.default_rng(7)

# 5 значимых + 45 случайных
true_positives = rng2.uniform(1e-6, 1e-3, 5)
nulls = rng2.uniform(0.05, 1.0, 45)
raw_pvals = np.concatenate([true_positives, nulls])

reject_bh, adj_bh = apply_fdr_bh(raw_pvals, alpha=0.05)
adj_bonf = apply_bonferroni(raw_pvals, alpha=0.05)

print(f'Всего тестов:        {len(raw_pvals)}')
print(f'BH FDR отвергнуто:   {reject_bh.sum()}')
print(f'Бонферрони значимых: {(adj_bonf < 0.05).sum()}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Raw p-values
ax = axes[0]
ax.hist(raw_pvals, bins=20, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(0.05, color='red', linestyle='--', label='α=0.05')
ax.set_title('Raw p-values', fontsize=12)
ax.set_xlabel('p-value')
ax.set_ylabel('Число тестов')
ax.legend()

# BH adjusted
ax = axes[1]
colors_bh = ['tomato' if r else 'steelblue' for r in reject_bh]
ax.scatter(range(len(adj_bh)), np.sort(adj_bh)[::-1], c=colors_bh, s=40, alpha=0.7)
ax.axhline(0.05, color='red', linestyle='--', label='α=0.05')
ax.set_title(f'BH FDR скорректированные\n({reject_bh.sum()} значимых)', fontsize=12)
ax.set_xlabel('Ранг теста')
ax.set_ylabel('Скорректированный p-value')
ax.legend()

# Сравнение
ax = axes[2]
ax.scatter(raw_pvals, adj_bh, s=40, alpha=0.6, color='steelblue', label='BH')
ax.scatter(raw_pvals, adj_bonf, s=40, alpha=0.4, color='orange', marker='s', label='Bonferroni')
ax.axhline(0.05, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Raw p-value')
ax.set_ylabel('Скорректированный p-value')
ax.set_title('Raw vs Adjusted p-values', fontsize=12)
ax.legend()

plt.suptitle('Коррекция на множественное тестирование', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. ETAS каталог: визуализация

In [ ]:
from src.analysis.etas_validation import ETASCatalogGenerator

gen = ETASCatalogGenerator(
    mu=0.02, K=0.15, alpha=1.0, c=0.01, p=1.1,
    d=1.0, q=1.5, m_min=6.5, b=1.0
)

etas_cat = gen.generate(
    n_background=80,
    t_end=50,
    max_local_radius_km=500,
    seed=42
)

print(f'ETAS каталог: {len(etas_cat)} событий')
print(f'  Фоновых:   {etas_cat["is_background"].sum()}')
print(f'  Дочерних:  {(~etas_cat["is_background"]).sum()}')
print(f'  Поколения: {etas_cat["generation"].value_counts().to_dict()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: время vs магнитуда
ax = axes[0]
bg = etas_cat[etas_cat['is_background']]
ch = etas_cat[~etas_cat['is_background']]
ax.scatter(bg['time_years'], bg['magnitude'], s=60, alpha=0.7, color='steelblue',
           label=f'Фоновые ({len(bg)})', edgecolors='navy', linewidths=0.5)
ax.scatter(ch['time_years'], ch['magnitude'], s=30, alpha=0.5, color='tomato',
           label=f'Дочерние ({len(ch)})', edgecolors='darkred', linewidths=0.3)
ax.set_xlabel('Время (лет)')
ax.set_ylabel('Магнитуда')
ax.set_title('ETAS каталог: время vs магнитуда', fontsize=12)
ax.legend()

# Карта: lat/lon с поколениями
ax = axes[1]
gen_colors = {0: 'steelblue', 1: 'tomato', 2: 'orange', 3: 'green', 4: 'purple'}
for gen_num in sorted(etas_cat['generation'].unique()):
    subset = etas_cat[etas_cat['generation'] == gen_num]
    c = gen_colors.get(gen_num, 'gray')
    label = 'Фоновые' if gen_num == 0 else f'Поколение {gen_num}'
    ax.scatter(subset['lon'], subset['lat'], s=40 * max(1, 3 - gen_num),
               alpha=0.6, color=c, label=label, edgecolors='k', linewidths=0.3)
ax.set_xlabel('Долгота')
ax.set_ylabel('Широта')
ax.set_title('Пространственное распределение по поколениям', fontsize=12)
ax.legend(loc='lower right', fontsize=9)

plt.suptitle('ETAS синтетический каталог', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Частота ложных серий на 10 ETAS каталогах

In [ ]:
from src.analysis.clustering import SeismicClusterAnalyzer

analyzer = SeismicClusterAnalyzer()

# Запускаем FPR-анализ на 10 каталогах (быстро)
fp_results = gen.run_false_positive_analysis(
    cluster_analyzer=analyzer,
    n_catalogs=10,
    seed=42
)

print('=== Результаты ETAS-валидации ===')
print(f'Частота ложных открытий (FPR):  {fp_results["false_positive_rate"]:.1%}')
print(f'Среднее число ложных серий:     {fp_results["mean_false_series"]:.2f}')
print(f'Серий по каталогам:             {fp_results["series_counts"]}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

counts = fp_results['series_counts']
cat_ids = np.arange(len(counts))
bars = ax.bar(cat_ids, counts, color=['tomato' if c > 0 else 'steelblue' for c in counts],
              edgecolor='k', alpha=0.8)
ax.axhline(fp_results['mean_false_series'], color='red', linestyle='--',
           label=f'Среднее = {fp_results["mean_false_series"]:.2f}')
ax.set_xticks(cat_ids)
ax.set_xticklabels([f'Кат. {i+1}' for i in cat_ids], rotation=30)
ax.set_ylabel('Число ложных серий')
ax.set_title(
    f'Ложные глобальные серии в ETAS каталогах\n'
    f'FPR = {fp_results["false_positive_rate"]:.0%}', fontsize=13
)
ax.legend()
plt.tight_layout()
plt.show()

## 5. Калибровка порогов: heatmap F1(N_min, n_regions)

In [ ]:
from src.analysis.threshold_calibration import calibrate_thresholds

calib_result = calibrate_thresholds(
    cluster_analyzer=analyzer,
    n_min_range=range(3, 7),
    n_regions_range=range(2, 5),
    n_synthetic=10,
    seed=42
)

print('Топ-5 комбинаций порогов (по F1):')
print(calib_result[['min_events', 'min_regions', 'precision', 'recall', 'f1']]
      .head(5).to_string(index=False))

In [ ]:
# Heatmap F1(N_min, n_regions)
pivot_f1 = calib_result.pivot(index='min_events', columns='min_regions', values='f1')
pivot_prec = calib_result.pivot(index='min_events', columns='min_regions', values='precision')
pivot_rec = calib_result.pivot(index='min_events', columns='min_regions', values='recall')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, pivot, title, cmap in zip(
    axes,
    [pivot_f1, pivot_prec, pivot_rec],
    ['F1-мера', 'Precision', 'Recall'],
    ['RdYlGn', 'Blues', 'Oranges']
):
    im = ax.imshow(pivot.values, aspect='auto', cmap=cmap, vmin=0, vmax=1)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel('n_regions (мин. регионов)')
    ax.set_ylabel('N_min (мин. событий)')
    ax.set_title(title, fontsize=12)
    plt.colorbar(im, ax=ax)
    # Аннотации
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=9,
                    color='black' if val < 0.7 else 'white')

plt.suptitle('Калибровка порогов N_min × n_regions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

best = calib_result.iloc[0]
print(f'\nРекомендуемые пороги: N_min={int(best["min_events"])}, n_regions={int(best["min_regions"])} (F1={best["f1"]:.3f})')

## 6. Тектоническое расстояние v2 с типами границ

In [ ]:
from src.analysis.tectonic_distance_v2 import TectonicDistanceV2, BOUNDARY_WEIGHTS

calc_v2 = TectonicDistanceV2()
calc_v2.build_graph(resolution_deg=1.0)

# Сравниваем расстояния через разные типы границ
test_pairs = [
    ('Токио → Сан-Франциско', 35.7, 139.7, 37.8, -122.4),
    ('Лима → Токио',          -12.0, -77.0, 35.7, 139.7),
    ('Анкоридж → Чили',       61.2, -149.9, -33.5, -70.5),
    ('Джакарта → Токио',      -6.2, 106.8, 35.7, 139.7),
]

print(f'{'Пара':40} {'Расстояние (км)':>18} {'Тип границы':>18}')
print('-' * 80)
for name, la1, lo1, la2, lo2 in test_pairs:
    dist = calc_v2.tectonic_distance_v2(la1, lo1, la2, lo2)
    btype = calc_v2.boundary_type_for_pair(la1, lo1, la2, lo2)
    print(f'{name:40} {dist:>18.0f} {btype:>18}')

In [ ]:
# Визуализация коэффициентов границ
fig, ax = plt.subplots(figsize=(8, 4))

types = list(BOUNDARY_WEIGHTS.keys())
weights = [BOUNDARY_WEIGHTS[t] for t in types]
colors = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4', '#9467bd']

bars = ax.barh(types, weights, color=colors, edgecolor='k', alpha=0.85)
ax.set_xlabel('Весовой коэффициент (меньше = эффективнее передача напряжений)')
ax.set_title('Коэффициенты типов границ плит\n(Tectonic Distance V2)', fontsize=12)
ax.axvline(1.0, color='red', linestyle=':', alpha=0.7, label='Базовый (субдукция)')

for bar, w in zip(bars, weights):
    ax.text(w + 0.01, bar.get_y() + bar.get_height()/2,
            f'{w:.1f}×', va='center', fontsize=11)

ax.legend()
ax.set_xlim(0, 1.8)
plt.tight_layout()
plt.show()

## Итоги

| Улучшение | Результат |
|-----------|----------|
| Gardner-Knopoff | Удаляет афтершоки, сохраняя главные толчки |
| Zaliapin-BZ | Автоматический порог по бимодальному распределению |
| FDR BH | Контролирует долю ложных открытий |
| ETAS валидация | Оценивает FPR на синтетических каталогах |
| Калибровка | Оптимизирует N_min и n_regions через F1 |
| TectonicDistance V2 | Взвешивает пути через тип границы плит |